In [ ]:
import pandas as pd
import numpy as np
import commons as c
import plotly.express as px
import plotly.graph_objects as go

# Get datasets

In [ ]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)

In [ ]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['true_label'] = np.where(df['true_label'] == True, "non-equivalent", "equivalent")
df['predicted_label'] = np.where(df['predicted_label'] == True, "non-equivalent", "equivalent")

# Get Box Plots

In [ ]:
def add_threshold_line(fig, t, cat_range, threshold_value, color):
    fig.add_shape(
        type="line",
        x0= min(cat_range),
        x1= max(cat_range),
        y0=threshold_value,
        y1=threshold_value,
        line=dict(color=color, dash="dash"),
        xref="x",
        yref="y",
    )
    
    # Add a dummy scatter trace for the legend
    fig.add_trace(go.Scatter(
            x=[None], y=[None],  # No actual points
            mode="lines",
            line=dict(color=color, dash="dash"),
            name=f"Threshold {t}",
            legendgroup="Thresholds",  # Group all thresholds together
            showlegend=True
    ))

In [ ]:
def print_box_plot(df, cat, hw, metric, output_folder, file_name):

    df = df.copy()  # Ensure it's a copy

    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['true_label'] = df['true_label'].map(label_mapping)
    
    # Create the box plot with the adjusted x-axis values
    fig = px.box(
        df, 
        y="noisy_distance", 
        x=cat, 
        color="true_label", 
        category_orders={
            cat: cat_range,
            "true_label": ["Equivalent mutant", "Non-Equivalent mutant"]
        },
        title="Boxplot of Distance by noise model and program type",
        labels={"Characteristic": "Characteristic", 'noisy_distance': "Distance", 'true_label': "Legend"},
        points=False,
        boxmode="group"
    ) 
    
    # Add the threshold line
    for t, color in zip(c.thresholds, px.colors.qualitative.Prism):
        threshold_value = c.get_tolerance_values(hw, t)[metric]
        add_threshold_line(fig, t, cat_range, threshold_value, color)
    
    # Adjust layout for better visualization
    fig.update_layout(
        xaxis=dict(tickmode="array", tickvals=cat_range)
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, f"Distance between the mutants and the oracle by {cat}", output_folder, file_name, yaxis_range=[0, 1])


In [ ]:
def category_plot(df, m, metric):
    
    for hw in c.hardware:
        df_hw = df[df['hardware'] == hw]
        df_metric = df_hw[df_hw['metric'] == m]
        
        for cat in ['Qubits_number', 'gates', 'depth'] : 
            selected_columns = df_metric[[cat, 'true_label', 'ideal_distance', 'noisy_distance']]  
            file_name = f'{hw}_{cat}'
            output_folder = f'results/RQ2/RQ2_1/{m}'
            print_box_plot(selected_columns, cat, hw, metric, output_folder, file_name)
            
        for cat in ['Algorithm', 'Input_type', 'Output_type'] : 
            selected_columns = df_metric[[cat, 'true_label', 'ideal_distance', 'noisy_distance']]  
            file_name = f'{hw}_{cat}'
            output_folder = f'results/RQ2/RQ2_2/{m}'
            print_box_plot(selected_columns, cat, hw, metric, output_folder, file_name)
            
        for cat in ['Relative_position', 'Gate_type', 'Operator'] : 
            selected_columns = df_metric[[cat, 'true_label', 'ideal_distance', 'noisy_distance']]  
            file_name = f'{hw}_{cat}'
            output_folder = f'results/RQ2/RQ2_3/{m}'
            print_box_plot(selected_columns, cat, hw, metric, output_folder, file_name)

In [ ]:
m = "T"
metric = "trace"
category_plot(df, m, metric)

m = "H"
metric = "hellinger"
category_plot(df, m, metric)